In [23]:
import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC

import requests
import pandas as pd
from datetime import date, timedelta
from functools import reduce
import os
from shutil import which
from bs4 import BeautifulSoup

chromedriver_autoinstaller.install()

'c:\\Python310\\lib\\site-packages\\chromedriver_autoinstaller\\157\\chromedriver.exe'

In [24]:
def _setup_chrome_options():
    """Set up Chrome options for local Selenium execution."""
    options = ChromeOptions()
    
    # Keep Chrome visible while running locally.
    # Uncomment this when you want headless execution again.
    # options.add_argument("--headless=new")
    
    options.add_argument("--start-maximized")
    
    return options

In [25]:
url = "https://www.rotowire.com/hockey/nhl-lineups.php"
# url = "https://www.rotowire.com/hockey/nhl-lineups.php?date=tomorrow"

options = _setup_chrome_options()
driver = webdriver.Chrome(options=options)

driver.set_page_load_timeout(20)
driver.set_script_timeout(20)

driver.get(url)

print("Page title:", driver.title)
print("Current URL:", driver.current_url)

Page title: NHL Starting Lineups: Hockey Daily Lineups | RotoWire
Current URL: https://www.rotowire.com/hockey/nhl-lineups.php


In [26]:
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, "lineups"))
)

print("Lineup container found!")

Lineup container found!


In [27]:
matchups = []

lineups = (
    driver.find_element(By.CLASS_NAME, "lineups")
    .find_elements(By.CLASS_NAME, "is-nhl")
)

print("Number of lineups found:", len(lineups))

Number of lineups found: 7


In [28]:
for lineup in lineups[:-1]:
    goalies = lineup.find_elements(
        By.CLASS_NAME,
        "lineup__player-highlight-name"
    )

    away_goalie = goalies[0].text.strip().split()[-1]
    home_goalie = goalies[1].text.strip().split()[-1]

    teams = lineup.find_element(
        By.CLASS_NAME,
        "lineup__matchup"
    ).find_elements(By.TAG_NAME, "a")

    if len(teams) >= 2:
        away_team = teams[0].text.split(" (")[0]
        home_team = teams[1].text.split(" (")[0]

        matchups.append(
            (away_team, home_team, away_goalie, home_goalie)
        )

matchups

[('Lightning', 'Sabres', 'Vasilevskiy', 'Luukkonen'),
 ('Rangers', 'Flames', 'Korpisalo', 'Cooley'),
 ('Flyers', 'Oilers', 'Vladar', 'Jarry'),
 ('Mammoth', 'Golden Knights', 'Cossa', 'Hart'),
 ('Predators', 'Ducks', 'Annunen', 'Brossoit'),
 ('Penguins', 'Sharks', 'Murashov', 'Askarov')]

In [29]:
driver.quit()

In [30]:
url = "https://moneypuck.com/goalies.htm"

options = _setup_chrome_options()
driver = webdriver.Chrome(options=options)

driver.set_page_load_timeout(20)
driver.set_script_timeout(20)

driver.get(url)

print("Page title:", driver.title)
print("Current URL:", driver.current_url)

Page title: NHL Goalie Advanced Stats 2026-2027
Current URL: https://moneypuck.com/goalies.htm


In [31]:
wait = WebDriverWait(driver, 10)

dropdown = wait.until(
    EC.presence_of_element_located((By.ID, "num_games_type"))
)

select = Select(dropdown)

select.select_by_value("_20")

print("Selected Last 20 Games")

Selected Last 20 Games


In [32]:
wait = WebDriverWait(driver, 10)

dropdown = wait.until(
    EC.presence_of_element_located((By.ID, "num_games_type"))
)

select = Select(dropdown)

select.select_by_value("_20")

print("Selected Last 20 Games")

Selected Last 20 Games


In [33]:
wait.until(
    EC.presence_of_element_located((By.ID, "goaliesTable"))
)

print("Goalie table found!")

Goalie table found!


In [34]:
table_html = driver.find_element(
    By.ID,
    "goaliesTable"
).get_attribute("outerHTML")

print("HTML length:", len(table_html))

HTML length: 180324


In [35]:
driver.quit()

In [36]:
soup = BeautifulSoup(table_html, "html.parser")

rows = soup.find_all("tr")[1:]

data = []

for tr in rows:
    tds = tr.find_all("td")

    if len(tds) < 7:
        continue

    try:
        rank = int(tds[0].get_text(strip=True))
        goalie = tds[2].get_text(strip=True).split()[-1]
        gsa = float(tds[6].get_text(strip=True))
    except:
        continue

    data.append({
        "Rank": rank,
        "Goalie": goalie,
        "GSAx": gsa
    })

goalie_df = pd.DataFrame(data)

goalie_df.head(20)

,Rank,Goalie,GSAx
0,1,Wallstedt,15.9
1,2,Fowler,12.3
2,3,Swayman,10.2
3,4,Saros,9.3
4,5,Kochetkov,8.8
5,6,Ullmark,8.3
6,7,Oettinger,7.6
7,8,Sorokin,7.3
8,9,Tarasov,6.5
9,10,Hofer,6.5


In [37]:
goalie_ranks = dict(
    zip(goalie_df["Goalie"], goalie_df["Rank"])
)

goalie_ranks

{'Wallstedt': 1,
 'Fowler': 2,
 'Swayman': 3,
 'Saros': 4,
 'Kochetkov': 5,
 'Ullmark': 6,
 'Oettinger': 7,
 'Sorokin': 8,
 'Tarasov': 9,
 'Hofer': 10,
 'Talbot': 11,
 'Nedeljkovic': 12,
 'Shesterkin': 13,
 'DiPietro': 14,
 'Varlamov': 15,
 'Woll': 16,
 'Askarov': 17,
 'Hill': 18,
 'Thompson': 19,
 'Cooley': 20,
 'Stolarz': 21,
 'Ellis': 22,
 'Hart': 23,
 'Garand': 24,
 'Markstrom': 25,
 'Levi': 26,
 'Brossoit': 27,
 'Hellebuyck': 28,
 'Murray': 29,
 'Blackwood': 30,
 'DeSmith': 31,
 'Tomkins': 32,
 'Korpisalo': 33,
 'Schmid': 34,
 'Forsberg': 35,
 'Skinner': 36,
 'Postava': 37,
 'Vasilevskiy': 38,
 'Lindgren': 39,
 'Sergeev': 40,
 'Grubauer': 41,
 'Vladar': 42,
 'Annunen': 43,
 'Black': 44,
 'Luukkonen': 45,
 'Martin': 46,
 'Hildeby': 47,
 'Jarry': 48,
 'Dostal': 49,
 'Murashov': 50,
 'Greaves': 51,
 'Daws': 52,
 'Reimer': 53,
 'Husso': 54,
 'Pickard': 55,
 'Knight': 56,
 'Merilinen': 57,
 'Bobrovsky': 58,
 'Silovs': 59,
 'Daccord': 60,
 'Ersson': 61,
 'Stevenson': 62,
 'Cossa': 63,
 

In [38]:
import os
from dotenv import load_dotenv

load_dotenv()

key = os.getenv("NST_API_KEY")

if not key:
    raise ValueError("NST_API_KEY not found. Check your .env file.")

In [40]:
from io import StringIO

# Date range
today = date.today()
start = today - timedelta(days=30)

# Your NST access key
key = "451fe70942969369f896204690d6cd68"

# Build the URL using NST's data subdomain
url = (
    "https://data.naturalstattrick.com/teamtable.php"
    f"?key={key}"
    f"&fromseason=20262027"
    f"&thruseason=20262027"
    f"&stype=2"
    f"&sit=5v5"
    f"&score=all"
    f"&rate=y"
    f"&team=all"
    f"&loc=B"
    f"&gpf=410"
    f"&fd={start}"
    f"&td={today}"
)

print("Requesting NST team table...")

response = requests.get(
    url,
    headers={"nst-key": key},
    timeout=30
)
response.raise_for_status()

html = response.text

print("Status code:", response.status_code)
print("HTML length:", len(html))
print("Contains table:", "<table" in html.lower())

# Parse the HTML table into a DataFrame
df = pd.read_html(
    StringIO(html),
    flavor="lxml",
    header=0,
    index_col=0,
    na_values=["-"]
)[0]

print("DataFrame shape:", df.shape)
df.head()


Requesting NST team table...
Status code: 200
HTML length: 64523
Contains table: True
DataFrame shape: (32, 71)


,Team,GP,TOI/GP,W,L,OTL,ROW,Points,Point %,CF/60,...,LDSA/60,LDSF%,LDGF/60,LDGA/60,LDGF%,LDSH%,LDSV%,SH%,SV%,PDO
1,Florida Panthers,14,44:53,8,6,0,8,16,0.571,56.70,...,11.36,48.93,0.48,0.38,55.56,4.39,96.64,7.91,88.04,0.960
2,Buffalo Sabres,13,47:08,6,4,3,6,15,0.577,57.57,...,9.40,57.14,0.69,0.39,63.64,5.47,95.83,10.27,89.62,0.999
3,Ottawa Senators,14,46:54,7,5,2,6,16,0.571,56.10,...,11.60,44.78,0.18,0.64,22.22,1.94,94.49,10.66,91.04,1.017
4,New Jersey Devils,15,50:37,11,4,0,10,22,0.733,61.46,...,10.82,54.33,0.71,0.24,75.00,5.52,97.81,10.47,90.67,1.011
5,Vegas Golden Knights,13,50:10,7,4,2,7,16,0.615,50.97,...,11.68,48.16,0.28,0.37,42.86,2.54,96.85,10.19,91.54,1.017


In [41]:
# Prepare team statistics
# NST's rate=y response provides the per-60 statistics directly.

if "Team" not in df.columns:
    df = df.reset_index()

rate_stats = ["CF/60", "CA/60", "xGF/60", "xGA/60"]
percentage_stats = ["CF%", "GF%", "xGF%", "HDCF%", "SH%"]

required_columns = ["Team"] + rate_stats + percentage_stats
missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing NST columns: {missing_columns}")

# Keep the original values
final_df = df[required_columns].copy()

# Rank per-60 metrics
# CA/60 and xGA/60: lower is better.
lower_is_better = ["CA/60", "xGA/60"]

for col in rate_stats:
    ascending = col in lower_is_better

    final_df[f"{col}_Rank"] = (
        pd.to_numeric(final_df[col], errors="coerce")
        .rank(method="min", ascending=ascending)
        .astype("Int64")
    )

# Rank percentage metrics (higher is better)
# Preserve the original naming convention:
# CF% = rank, CF%_1 = actual percentage.

for col in percentage_stats:
    final_df[f"{col}_1"] = final_df[col]

    final_df[col] = (
        pd.to_numeric(final_df[col], errors="coerce")
        .rank(method="min", ascending=False)
        .astype("Int64")
    )


In [42]:
final_df

,Team,CF/60,CA/60,xGF/60,xGA/60,CF%,GF%,xGF%,HDCF%,SH%,CF/60_Rank,CA/60_Rank,xGF/60_Rank,xGA/60_Rank,CF%_1,GF%_1,xGF%_1,HDCF%_1,SH%_1
1,Florida Panthers,56.70,50.50,2.61,2.45,5,29,13,13,29,14,6,19,8,52.89,40.00,51.58,51.36,7.91
2,Buffalo Sabres,57.57,53.36,2.98,2.66,8,11,9,20,9,11,7,3,17,51.90,52.63,52.83,50.43,10.27
3,Ottawa Senators,56.10,54.18,2.67,2.62,11,8,14,22,7,16,10,17,14,50.87,54.72,50.46,48.64,10.66
4,New Jersey Devils,61.46,48.90,3.04,2.24,2,6,2,2,8,3,2,2,4,55.69,58.82,57.58,56.45,10.47
5,Vegas Golden Knights,50.97,58.14,2.44,2.75,27,9,25,23,10,28,26,25,21,46.71,54.00,47.01,47.95,10.19
6,Minnesota Wild,58.56,55.86,2.88,2.46,10,3,6,9,12,7,15,8,10,51.18,66.00,53.98,52.65,10.09
7,Detroit Red Wings,54.74,55.62,2.45,2.54,17,16,21,18,27,21,13,24,13,49.60,48.08,49.13,51.05,8.47
8,Columbus Blue Jackets,53.78,55.72,2.46,3.01,21,23,26,24,31,23,14,23,28,49.12,44.74,45.00,47.14,7.02
9,Montreal Canadiens,59.39,55.50,2.84,2.93,9,4,20,19,4,5,12,9,24,51.69,62.26,49.23,50.62,11.96
10,Washington Capitals,56.96,58.45,2.71,2.94,18,10,24,25,3,13,29,15,26,49.35,52.83,47.90,46.31,12.12


In [43]:
def normalize_team(name):
    return (
        name.lower()
        .replace(".", "")
        .replace("é", "e")
        .replace(" ", "")
    )

In [44]:
res = pd.DataFrame()

for away, home, away_goalie, home_goalie in matchups:

    away_df = final_df[
        final_df["Team"].str.contains(
            away,
            case=False,
            na=False
        )
    ]

    home_df = final_df[
        final_df["Team"].str.contains(
            home,
            case=False,
            na=False
        )
    ]

    away_rank = goalie_ranks.get(away_goalie)
    home_rank = goalie_ranks.get(home_goalie)

    away_df = away_df.assign(
        Goalie=away_goalie,
        Goalie_Rank=away_rank
    )

    home_df = home_df.assign(
        Goalie=home_goalie,
        Goalie_Rank=home_rank
    )

    matchup_df = pd.concat(
        [away_df, home_df],
        ignore_index=True
    )

    res = pd.concat(
        [res, matchup_df],
        ignore_index=True
    )

In [45]:
output_file = "../public/result.csv"

res.to_csv(output_file, index=False)

print(f"Saved {len(res)} rows to {output_file}")

Saved 12 rows to ../public/result.csv
